In [1]:
import pandas as pd
import numpy as np
import rich
import matplotlib.pyplot as plt
import sklearn
import torch
import optuna

from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from rich.progress import track

In [2]:
"""
HALT preprocessing pipeline.

Loads the CSS2_UQ dataset from HuggingFace, validates and filters rows,
and computes (N, T_max, 25) feature sequences with binary is_correct labels.

Called from training code, e.g.:
    from UQ.halt.preprocessing.preprocess_halt import preprocess
    features, labels = preprocess()

Returns:
    features: float32 array of shape (N, MAX_LEN, 25)
    labels:   float32 array of shape (N,) — binary is_correct (0 or 1)
"""

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

HF_DATASET  = "auhsoJ69/mmlu-pro-traces"
MAX_LEN     = 192
TOP_K       = 20
FEATURE_DIM = 5 + TOP_K   # 5 engineered + 20 raw log-probs

# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------

def compute_step_features(logprobs_step: np.ndarray):
    """
    Compute 5 engineered uncertainty features for one token step.

    Args:
        logprobs_step: (20,) top-20 log-probs; index 0 = selected token

    Returns:
        feats: (5,) [avg_logprob, rank_proxy, h_overall, h_alts, h_dec]
        h_dec: scalar decision entropy (needed externally for delta)
    """
    eps = 1e-10

    # Numerically stable truncated softmax
    m = np.max(logprobs_step)
    probs = np.exp(logprobs_step - m)
    probs = probs / (probs.sum() + eps)

    # 1. Average log-probability
    avg_logprob = float(np.mean(logprobs_step))

    # 2. Rank proxy of selected token
    rank_proxy = float(1 + np.sum(logprobs_step[1:] > logprobs_step[0]))

    # 3. Overall entropy over top-k distribution
    h_overall = float(-np.sum(probs * np.log(probs + eps)))

    # 4. Alternatives-only entropy
    alts_probs = probs[1:]
    alts_probs = alts_probs / (alts_probs.sum() + eps)
    h_alts = float(-np.sum(alts_probs * np.log(alts_probs + eps)))

    # 5. Binary decision entropy (log-domain for numerical stability)
    best_alt_lp = float(np.max(logprobs_step[1:]))
    log_sum = np.logaddexp(float(logprobs_step[0]), best_alt_lp)
    log_pc = float(logprobs_step[0]) - log_sum
    log_1_pc = best_alt_lp - log_sum
    pc = float(np.clip(np.exp(log_pc), eps, 1 - eps))
    h_dec = float(-(pc * log_pc + (1 - pc) * log_1_pc))

    feats = np.array([avg_logprob, rank_proxy, h_overall, h_alts, h_dec], dtype=np.float32)
    return feats, h_dec


def build_feature_sequence(top20_logprobs) -> np.ndarray | None:
    """
    Build the padded (MAX_LEN, 25) feature matrix for one example.

    Returns None if the sequence is invalid and should be dropped.

    Feature layout per timestep:
        [avg_logprob, rank_proxy, h_overall, h_alts, delta_h_dec, lp_0..lp_19]
    """
    if top20_logprobs is None or len(top20_logprobs) == 0:
        return None

    padded = np.zeros((MAX_LEN, FEATURE_DIM), dtype=np.float32)
    prev_h_dec = 0.0
    valid_steps = 0

    for t, step in enumerate(top20_logprobs):
        if t >= MAX_LEN:
            break
        if step is None:
            continue

        step = np.array(step, dtype=np.float32)

        if step.ndim != 1 or len(step) == 0:
            continue
        if np.any(np.isnan(step)) or np.any(np.isinf(step)):
            continue
        if len(step) < TOP_K:
            step = np.pad(step, (0, TOP_K - len(step)), constant_values=-1e9)
        step = step[:TOP_K]

        stat_feats, h_dec = compute_step_features(step)

        delta_h_dec = h_dec - prev_h_dec
        stat_feats[4] = delta_h_dec
        prev_h_dec = h_dec

        padded[t, :5] = stat_feats
        padded[t, 5:] = step
        valid_steps += 1

    if valid_steps == 0:
        return None

    return padded


# ---------------------------------------------------------------------------
# Row validation
# ---------------------------------------------------------------------------

def validate_row(row) -> tuple[bool, str]:
    """
    Check a dataset row is usable for training.

    Returns (is_valid, reason_if_invalid).
    """
    if not row.get("parse_success", False):
        return False, "parse_success=False"
    if row.get("top20_token_logprobs") is None or len(row["top20_token_logprobs"]) == 0:
        return False, "empty top20_token_logprobs"
    if row.get("is_correct") is None:
        return False, "missing is_correct"
    return True, ""


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def preprocess(hf_dataset: str = HF_DATASET) -> tuple[np.ndarray, np.ndarray]:
    """
    Load, validate, and featurize the CSS2_UQ dataset.

    Returns:
        features: float32 array of shape (N, MAX_LEN, 25)
        labels:   float32 array of shape (N,) — binary is_correct (0 or 1)
    """
    print(f"Loading dataset: {hf_dataset}")
    ds = load_dataset(hf_dataset, data_files="examples.parquet")
    df = ds["train"].to_pandas()
    print(f"Total rows: {len(df)}")

    features_list = []
    labels_list   = []
    skipped       = {}

    for _, row in df.iterrows():
        is_valid, reason = validate_row(row)
        if not is_valid:
            skipped[reason] = skipped.get(reason, 0) + 1
            continue

        features = build_feature_sequence(row["top20_token_logprobs"])
        if features is None:
            skipped["bad feature sequence"] = skipped.get("bad feature sequence", 0) + 1
            continue

        features_list.append(features)
        labels_list.append(float(row["is_correct"]))

    n = len(features_list)
    print(f"Rows kept:    {n}")
    print(f"Rows skipped: {sum(skipped.values())}")
    for reason, count in skipped.items():
        if count > 0:
            print(f"  {reason}: {count}")

    if n == 0:
        raise RuntimeError("No valid rows found — check dataset and filters.")

    features_arr = np.stack(features_list, axis=0)       # (N, MAX_LEN, 25)
    labels_arr   = np.array(labels_list, dtype=np.int8)  # (N,)

    n_correct   = int(labels_arr.sum())
    n_incorrect = n - n_correct
    print(f"Label balance — correct: {n_correct} | incorrect: {n_incorrect}")
    print(f"Feature matrix shape: {features_arr.shape}")

    return features_arr, labels_arr

In [3]:
features, labels = preprocess()

Loading dataset: auhsoJ69/mmlu-pro-traces


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total rows: 12032
Rows kept:    11424
Rows skipped: 608
  parse_success=False: 608
Label balance — correct: 6793 | incorrect: 4631
Feature matrix shape: (11424, 192, 25)


In [4]:
features.shape, labels.shape

((11424, 192, 25), (11424,))

In [5]:
train_features, temp_features, train_labels, temp_labels = train_test_split(features, labels, train_size=0.7, test_size=0.3, random_state=42, stratify=labels)
val_features, test_features, val_labels, test_labels = train_test_split(temp_features, temp_labels, train_size=0.5, test_size=0.5, random_state=42, stratify=temp_labels)

In [22]:
train_features.shape, train_labels.shape

((7996, 192, 25), (7996,))

In [23]:
val_features.shape, val_labels.shape

((1714, 192, 25), (1714,))

In [24]:
test_features.shape, test_labels.shape

((1714, 192, 25), (1714,))

In [25]:
train_dataset = TensorDataset(torch.from_numpy(train_features), torch.from_numpy(train_labels))
val_dataset = TensorDataset(torch.from_numpy(val_features), torch.from_numpy(val_labels))
test_dataset = TensorDataset(torch.from_numpy(test_features), torch.from_numpy(test_labels))

train_dataloader = DataLoader(train_dataset, batch_size=32)
val_dataloader = DataLoader(val_dataset, batch_size=32)
test_dataloader = DataLoader(test_dataset, batch_size=32)

In [26]:
class ConfidenceLSTM(nn.Module):
    def __init__(self, state_feature_size, hidden_size, input_size=25, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, state_feature_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(state_feature_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        output = self.fc1(h_n[-1])
        output = self.relu(output)
        output = self.fc2(output)
        return output

In [39]:
def train_model(model, learning_rate: float, save_weights: bool=True) -> float:
  ''' returns BEST_VAL_LOSS for optuna optimization '''
  EPOCH = 100
  RANDOM_SEED = 0
  NO_IMPROVE = 0
  PATIENCE = 10 # Patience for early stopping
  BEST_VAL_LOSS = float("inf")
  torch.manual_seed(RANDOM_SEED)
  torch.cuda.manual_seed(RANDOM_SEED)
  np.random.seed(RANDOM_SEED)
  device = "cuda" if torch.cuda.is_available() else "cpu"

  model.to(device)
  LOSS_FUNC = nn.BCEWithLogitsLoss()
  OPTIMIZER = torch.optim.Adam(model.parameters(), lr=learning_rate)
  SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=OPTIMIZER, mode="min", factor=0.5, patience=5)

  for epoch in range(EPOCH):
    model.train()
    val_loss = 0

    # ------------------------TRAINING------------------------------------
    for features, labels in train_dataloader:
      features = features.to(device)
      labels = labels.float().to(device)
      y_logits = model(features)
      loss = LOSS_FUNC(y_logits, labels.unsqueeze(dim=1))

      OPTIMIZER.zero_grad()
      loss.backward()
      OPTIMIZER.step()

    # -----------------------------VALIDATING-------------------------------
    model.eval()
    with torch.inference_mode():
      for features, labels in val_dataloader:
        features = features.to(device)
        labels = labels.float().to(device)
        y_logits = model(features)
        loss = LOSS_FUNC(y_logits, labels.unsqueeze(dim=1))
        val_loss += loss.item()

    SCHEDULER.step(val_loss)

    # ----------------------EARLY STOPPING------------------------
    if val_loss < BEST_VAL_LOSS:
      BEST_VAL_LOSS = val_loss
      NO_IMPROVE = 0

      if save_weights:
        torch.save(model.state_dict(), "best_weights.pth")
    else:
      NO_IMPROVE += 1

    if NO_IMPROVE >= PATIENCE:
      break

  return BEST_VAL_LOSS

In [28]:
def objective(trial):
  state_feature_size = trial.suggest_int('state_feature_size', 32, 128)
  learning_rate = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
  hidden_size = trial.suggest_int('hidden_size', 32, 64)

  UQ_model = ConfidenceLSTM(state_feature_size=state_feature_size, hidden_size=hidden_size)
  best_val_loss = train_model(UQ_model, learning_rate, save_weights=False)

  return best_val_loss

In [29]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print("Best Hyperparameters:", study.best_params)

[I 2026-05-13 04:29:50,759] A new study created in memory with name: no-name-197fa732-7a46-4749-af18-a12df57fb0a8
[I 2026-05-13 04:31:59,483] Trial 0 finished with value: 36.31486213207245 and parameters: {'state_feature_size': 92, 'lr': 1.335588900386364e-05, 'hidden_size': 48}. Best is trial 0 with value: 36.31486213207245.
[I 2026-05-13 04:33:02,978] Trial 1 finished with value: 34.96881026029587 and parameters: {'state_feature_size': 57, 'lr': 0.000831486491032394, 'hidden_size': 39}. Best is trial 1 with value: 34.96881026029587.
[I 2026-05-13 04:33:57,781] Trial 2 finished with value: 35.57236593961716 and parameters: {'state_feature_size': 38, 'lr': 7.718473279908301e-05, 'hidden_size': 59}. Best is trial 1 with value: 34.96881026029587.
[I 2026-05-13 04:36:13,396] Trial 3 finished with value: 34.968126237392426 and parameters: {'state_feature_size': 98, 'lr': 0.0001842357890179435, 'hidden_size': 52}. Best is trial 3 with value: 34.968126237392426.
[I 2026-05-13 04:38:20,152] T

Best Hyperparameters: {'state_feature_size': 125, 'lr': 0.00036758452542195715, 'hidden_size': 56}


In [40]:
# Build model with best hyperparameters
best_params = study.best_params
UQ_model = ConfidenceLSTM(state_feature_size=best_params['state_feature_size'], hidden_size=best_params['hidden_size'])
learning_rate = best_params['lr']
train_model(UQ_model, learning_rate, save_weights=True)

34.34813046455383

In [45]:
# Load best weights
device = "cuda" if torch.cuda.is_available() else "cpu"
UQ_model = ConfidenceLSTM(state_feature_size=best_params['state_feature_size'], hidden_size=best_params['hidden_size'])
UQ_model.load_state_dict(torch.load('/content/best_weights.pth', weights_only=True))
UQ_model.to(device)

ConfidenceLSTM(
  (lstm): LSTM(25, 125, batch_first=True)
  (fc1): Linear(in_features=125, out_features=56, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=56, out_features=1, bias=True)
)

In [46]:
def evaluate_model(model, dataloader) -> float:
  ground_truths = []
  prediction_probs = []

  model.eval()
  with torch.inference_mode():
      for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        y_logits = UQ_model(features)
        y_probs = torch.sigmoid(y_logits)
        ground_truths.append(labels.cpu())
        prediction_probs.append(y_probs.cpu())

      ground_truths = torch.concat(ground_truths).numpy()
      prediction_probs = torch.concat(prediction_probs).numpy()
      brier_score = mean_squared_error(ground_truths, prediction_probs)
      return brier_score

In [47]:
brier_score_train = evaluate_model(UQ_model, train_dataloader)
brier_score_train

0.22463274002075195

In [48]:
brier_score_val = evaluate_model(UQ_model, val_dataloader)
brier_score_val

0.222415030002594

In [49]:
brier_score_test = evaluate_model(UQ_model, test_dataloader)
brier_score_test

0.22739195823669434